# Exploración y limpieza de datos — Retrasos en Aerolíneas 2015
Preparamos los datos del DOT (Department of Transportation) para responder: **¿por qué se retrasan y cancelan los vuelos?**  
El resultado son cuatro tablas limpias listas para Power BI.

In [20]:
import pandas as pd
import os

flights = pd.read_csv('../data/raw/flights.csv')
airlines = pd.read_csv('../data/raw/airlines.csv')
airports = pd.read_csv('../data/raw/airports.csv')

C:\Users\gerar\AppData\Local\Temp\ipykernel_9944\1336779955.py:4: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv('../data/raw/flights.csv')


## 1. Exploración inicial
Revisamos dimensiones, columnas y valores nulos del dataset crudo para entender con qué datos contamos antes de limpiar.

In [21]:
flights.shape
flights.columns
flights.isnull().sum().sort_values(ascending=False)

CANCELLATION_REASON    5729195
LATE_AIRCRAFT_DELAY    4755640
WEATHER_DELAY          4755640
AIRLINE_DELAY          4755640
AIR_SYSTEM_DELAY       4755640
SECURITY_DELAY         4755640
ELAPSED_TIME            105071
AIR_TIME                105071
ARRIVAL_DELAY           105071
WHEELS_ON                92513
TAXI_IN                  92513
ARRIVAL_TIME             92513
WHEELS_OFF               89047
TAXI_OUT                 89047
DEPARTURE_TIME           86153
DEPARTURE_DELAY          86153
TAIL_NUMBER              14721
SCHEDULED_TIME               6
DESTINATION_AIRPORT          0
DAY                          0
DAY_OF_WEEK                  0
AIRLINE                      0
FLIGHT_NUMBER                0
MONTH                        0
ORIGIN_AIRPORT               0
SCHEDULED_DEPARTURE          0
YEAR                         0
SCHEDULED_ARRIVAL            0
DISTANCE                     0
DIVERTED                     0
CANCELLED                    0
dtype: int64

In [22]:
flights[flights["ARRIVAL_DELAY"].isnull()][["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
1          0           89884
0          1           15187
Name: count, dtype: int64

## 2. Limpieza y enriquecimiento
Construimos `flights_clean_all`: eliminamos columnas irrelevantes, clasificamos cada vuelo, normalizamos causas de retraso y cancelación, corregimos nulos y añadimos columnas de tiempo para Power BI.

In [23]:
def classify_flight(row):
    if row["CANCELLED"] == 1:
        return "CANCELLED"
    elif row["DIVERTED"] == 1:
        return "DIVERTED"
    elif row["ARRIVAL_DELAY"] >= 15:
        return "DELAYED"
    else:
        return "ON TIME / EARLY"

In [24]:
flights_clean_all = flights.copy()

In [25]:
flights_clean_all.drop(columns=["FLIGHT_NUMBER", "TAIL_NUMBER", "WHEELS_OFF", "WHEELS_ON", "TAXI_OUT", "TAXI_IN"], inplace=True)

In [26]:
flights_clean_all["FLIGHT_STATUS"] = flights_clean_all.apply(classify_flight, axis =1)

In [27]:
delay_causes = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

flights_clean_all[delay_causes] = flights_clean_all[delay_causes].fillna(0)
flights_clean_all["TOTAL_DELAY_CAUSES"] = flights_clean_all[delay_causes].sum(axis=1)

flights_clean_all["CANCELLATION_REASON"] = flights_clean_all["CANCELLATION_REASON"].fillna("Not cancelled")

reason_map = {
    "A": "Carrier",
    "B": "Weather",
    "C": "National Air System",
    "D": "Security",
}

flights_clean_all["CANCELLATION_REASON"] = flights_clean_all["CANCELLATION_REASON"].map(reason_map)


In [28]:
# SCHEDULED_DEPARTURE and SCHEDULED_ARRIVAL are int64 in HHMM format (e.g. 1430 = 14:30).
# Convert to minutes before subtracting to get a correct duration.
# % 1440 handles overnight flights where arrival rolls past midnight.

dep_min = (flights_clean_all["SCHEDULED_DEPARTURE"] // 100) * 60 + (flights_clean_all["SCHEDULED_DEPARTURE"] % 100)
arr_min = (flights_clean_all["SCHEDULED_ARRIVAL"] // 100) * 60 + (flights_clean_all["SCHEDULED_ARRIVAL"] % 100)

diff = (arr_min - dep_min) % 1440  # result is int64

mask = flights_clean_all["SCHEDULED_TIME"].isnull()
flights_clean_all.loc[mask, "SCHEDULED_TIME"] = diff[mask].astype("float64")

print(f"Nulls remaining in SCHEDULED_TIME: {flights_clean_all['SCHEDULED_TIME'].isnull().sum()}")

Nulls remaining in SCHEDULED_TIME: 0


In [29]:
# 1. Proper date column for Power BI time intelligence
flights_clean_all["DATE"] = pd.to_datetime(flights_clean_all[["YEAR", "MONTH", "DAY"]])

# 2. Day of week label (DOT convention: 1=Monday, 7=Sunday)
dow_map = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday",
           5: "Friday", 6: "Saturday", 7: "Sunday"}
flights_clean_all["DAY_OF_WEEK_LABEL"] = flights_clean_all["DAY_OF_WEEK"].map(dow_map)

# 3. Departure hour for time-of-day analysis
flights_clean_all["DEPARTURE_HOUR"] = flights_clean_all["SCHEDULED_DEPARTURE"] // 100

In [30]:
# --- Verify new columns ---

# 1. DATE: Jan 1 2015 should be a Thursday — confirms DAY_OF_WEEK=4 maps correctly
sample = flights_clean_all[["YEAR", "MONTH", "DAY", "DATE", "DAY_OF_WEEK", "DAY_OF_WEEK_LABEL"]].head(3)
print(sample)

# 2. DAY_OF_WEEK_LABEL distribution — should show 7 days with no nulls
print("\nDAY_OF_WEEK_LABEL value counts:")
print(flights_clean_all["DAY_OF_WEEK_LABEL"].value_counts().sort_index())

# 3. DEPARTURE_HOUR range — should be 0 to 23
print(f"\nDEPARTURE_HOUR range: {flights_clean_all['DEPARTURE_HOUR'].min()} - {flights_clean_all['DEPARTURE_HOUR'].max()}")
print(flights_clean_all["DEPARTURE_HOUR"].value_counts().sort_index())

   YEAR  MONTH  DAY       DATE  DAY_OF_WEEK DAY_OF_WEEK_LABEL
0  2015      1    1 2015-01-01            4          Thursday
1  2015      1    1 2015-01-01            4          Thursday
2  2015      1    1 2015-01-01            4          Thursday

DAY_OF_WEEK_LABEL value counts:
DAY_OF_WEEK_LABEL
Friday       862209
Monday       865543
Saturday     700545
Sunday       817764
Thursday     872521
Tuesday      844600
Wednesday    855897
Name: count, dtype: int64

DEPARTURE_HOUR range: 0 - 23
DEPARTURE_HOUR
0      14664
1       5159
2       1414
3        778
4        531
5     118051
6     406940
7     393947
8     381014
9     351403
10    371644
11    358084
12    355611
13    363509
14    329715
15    367760
16    334153
17    390362
18    334380
19    331338
20    259432
21    187467
22    117551
23     44172
Name: count, dtype: int64


## 3. Subconjuntos por estado de vuelo
Dividimos `flights_clean_all` en tres tablas según el resultado del vuelo, conservando solo las columnas que aplican a cada caso.

In [31]:
flights_arrived = flights_clean_all[(flights_clean_all["CANCELLED"] == 0) & (flights_clean_all["DIVERTED"] == 0)].copy()
flights_arrived.drop(columns=["CANCELLATION_REASON",], inplace=True)

flights_arrived.isnull().sum().sort_values(ascending=False)

YEAR                   0
MONTH                  0
DAY                    0
DAY_OF_WEEK            0
AIRLINE                0
ORIGIN_AIRPORT         0
DESTINATION_AIRPORT    0
SCHEDULED_DEPARTURE    0
DEPARTURE_TIME         0
DEPARTURE_DELAY        0
SCHEDULED_TIME         0
ELAPSED_TIME           0
AIR_TIME               0
DISTANCE               0
SCHEDULED_ARRIVAL      0
ARRIVAL_TIME           0
ARRIVAL_DELAY          0
DIVERTED               0
CANCELLED              0
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
FLIGHT_STATUS          0
TOTAL_DELAY_CAUSES     0
DATE                   0
DAY_OF_WEEK_LABEL      0
DEPARTURE_HOUR         0
dtype: int64

In [32]:
flights_cancelled = flights_clean_all[flights_clean_all["CANCELLED"] == 1].copy()

# Como el vuelo es cancelado, elimino las columnas que dependen de la llegada del vuelo, ya que no aplica.
cancelled_drop_cols = [
    "DEPARTURE_TIME",
    "DEPARTURE_DELAY",
    "ELAPSED_TIME",
    "AIR_TIME",
    "ARRIVAL_TIME",
    "ARRIVAL_DELAY",
    "DIVERTED",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
    "TOTAL_DELAY_CAUSES"
]


flights_cancelled = flights_cancelled.drop(columns=cancelled_drop_cols)
flights_cancelled.isnull().sum().sort_values(ascending=False)

flights_cancelled[flights_cancelled["CANCELLATION_REASON"] != "Not cancelled"].head(10)


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,CANCELLED,CANCELLATION_REASON,FLIGHT_STATUS,DATE,DAY_OF_WEEK_LABEL,DEPARTURE_HOUR
32,2015,1,1,4,AS,ANC,SEA,135,205.0,1448,600,1,Carrier,CANCELLED,2015-01-01,Thursday,1
42,2015,1,1,4,AA,PHX,DFW,200,120.0,868,500,1,Weather,CANCELLED,2015-01-01,Thursday,2
68,2015,1,1,4,OO,MAF,IAH,510,87.0,429,637,1,Weather,CANCELLED,2015-01-01,Thursday,5
82,2015,1,1,4,MQ,SGF,DFW,525,95.0,364,700,1,Weather,CANCELLED,2015-01-01,Thursday,5
90,2015,1,1,4,OO,RDD,SFO,530,90.0,199,700,1,Carrier,CANCELLED,2015-01-01,Thursday,5
128,2015,1,1,4,MQ,CHS,DFW,545,190.0,987,755,1,Weather,CANCELLED,2015-01-01,Thursday,5
131,2015,1,1,4,OO,SMX,LAX,545,66.0,134,651,1,Carrier,CANCELLED,2015-01-01,Thursday,5
147,2015,1,1,4,MQ,ABI,DFW,550,55.0,158,645,1,Weather,CANCELLED,2015-01-01,Thursday,5
166,2015,1,1,4,MQ,XNA,DFW,555,75.0,280,710,1,Weather,CANCELLED,2015-01-01,Thursday,5
206,2015,1,1,4,AA,DCA,DFW,600,215.0,1192,835,1,Weather,CANCELLED,2015-01-01,Thursday,6


In [33]:
flights_diverted = flights_clean_all[flights_clean_all["DIVERTED"] == 1].copy()

diverted_drop_cols = [
   "ELAPSED_TIME",
    "AIR_TIME",
    "ARRIVAL_DELAY",
    "ARRIVAL_TIME",
    "CANCELLED",
    "CANCELLATION_REASON",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

flights_diverted = flights_diverted.drop(columns=diverted_drop_cols)
flights_diverted.isnull().sum().sort_values(ascending=False)

flights_diverted[flights_diverted["SCHEDULED_TIME"].isnull()].head(10)

flights_diverted.info()


<class 'pandas.DataFrame'>
Index: 15187 entries, 724 to 5818160
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   YEAR                 15187 non-null  int64         
 1   MONTH                15187 non-null  int64         
 2   DAY                  15187 non-null  int64         
 3   DAY_OF_WEEK          15187 non-null  int64         
 4   AIRLINE              15187 non-null  str           
 5   ORIGIN_AIRPORT       15187 non-null  object        
 6   DESTINATION_AIRPORT  15187 non-null  object        
 7   SCHEDULED_DEPARTURE  15187 non-null  int64         
 8   DEPARTURE_TIME       15187 non-null  float64       
 9   DEPARTURE_DELAY      15187 non-null  float64       
 10  SCHEDULED_TIME       15187 non-null  float64       
 11  DISTANCE             15187 non-null  int64         
 12  SCHEDULED_ARRIVAL    15187 non-null  int64         
 13  DIVERTED             15187 non-null  int64 

## 4. Validación de tablas de referencia
Verificamos que `airlines` y `airports` no tienen duplicados en `IATA_CODE`, la llave que usaremos para relacionar tablas en Power BI.

In [34]:
airports[airports["IATA_CODE"].duplicated(keep=False)]

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE


In [35]:
airlines[airlines["IATA_CODE"].duplicated(keep=False)]

,IATA_CODE,AIRLINE


## 5. Validación final
Confirmamos que los tamaños de cada tabla son correctos, que los estados de vuelo suman el total y que no hay contaminación entre categorías.

In [36]:
# 1. Tamaños finales
print("All:", flights_clean_all.shape)
print("Arrived:", flights_arrived.shape)
print("Cancelled:", flights_cancelled.shape)
print("Diverted:", flights_diverted.shape)
print("Airlines:", airlines.shape)
print("Airports:", airports.shape)

# 2. Validar que los estados suman el total
print(flights_clean_all["FLIGHT_STATUS"].value_counts())

# 3. Validar llaves de relación
print(airlines["IATA_CODE"].duplicated().sum())
print(airports["IATA_CODE"].duplicated().sum())

flights_clean_all[
    (flights_clean_all["FLIGHT_STATUS"] == "DELAYED") &
    (
        (flights_clean_all["CANCELLED"] == 1) |
        (flights_clean_all["DIVERTED"] == 1)
    )
].shape

All: (5819079, 30)
Arrived: (5714008, 29)
Cancelled: (89884, 17)
Diverted: (15187, 21)
Airlines: (14, 2)
Airports: (322, 7)
FLIGHT_STATUS
ON TIME / EARLY    4650569
DELAYED            1063439
CANCELLED            89884
DIVERTED             15187
Name: count, dtype: int64
0
0


(0, 30)

## 6. Exportar datos procesados
Guardamos los datasets limpios en `data/processed/` listos para importar en Power BI.

In [38]:
flights_clean_all.to_csv("../data/processed/flights_clean_all.csv", index=False)
flights_arrived.to_csv("../data/processed/flights_arrived.csv", index=False)
flights_cancelled.to_csv("../data/processed/flights_cancelled.csv", index=False)
flights_diverted.to_csv("../data/processed/flights_diverted.csv", index=False)

airlines.to_csv("../data/processed/airlines_clean.csv", index=False)
airports.to_csv("../data/processed/airports_clean.csv", index=False)

os.listdir("../data/processed")

['airlines_clean.csv',
 'airports_clean.csv',
 'flights_arrived.csv',
 'flights_cancelled.csv',
 'flights_clean_all.csv',
 'flights_diverted.csv']